# Handling `DoseResponseData`

This notebook stays at the data-container layer. It shows how to build, inspect, subset, merge, and serialize `DoseResponseData` objects before any curve fitting happens.

In [ ]:
import pandas as pd

import bindcurve as bc

The following pandas settings are only for notebook display.

In [ ]:
pd.set_option("display.precision", 2)
pd.set_option("display.width", 200)

## Build a small long-form dataset

We will make one primary dataset with three compounds. The rows are intentionally not ordered alphabetically, so we can later see that `data.compounds` uses sorted compound IDs for index-based selection.

In [ ]:
def make_rows(compound_id, experiment_id, response_map):
    rows = []
    for concentration, responses in response_map.items():
        for replicate_index, response in enumerate(responses, start=1):
            rows.append(
                {
                    "compound_id": compound_id,
                    "experiment_id": experiment_id,
                    "concentration": concentration,
                    "replicate_id": f"rep{replicate_index}",
                    "response": response,
                }
            )
    return rows

raw_primary = pd.DataFrame(
    make_rows("cmpd_b", "exp1", {0.1: [96.0, 95.0], 1.0: [70.0, 69.0]})
    + make_rows("cmpd_a", "exp1", {0.1: [92.0, 91.0], 1.0: [48.0, 47.0]})
    + make_rows("cmpd_c", "exp1", {0.1: [98.0, 97.0], 1.0: [82.0, 81.0]})
)

data = bc.DoseResponseData.from_dataframe(
    raw_primary,
    metadata={"source": "primary"},
)

data.table

## Inspect the container

`summary()` gives a compact compound-level overview, while `compounds` exposes the sorted compound IDs used for index-based selection.

In [ ]:
print(data.compounds)
data.summary()

You can still drill into one compound with `select_compound()` and inspect the arithmetic replicate average that BindCurve uses inside the canonical fitting pipeline.

In [ ]:
compound_view = data.select_compound("cmpd_a")
compound_view.aggregate_replicates()

## Keep only selected compounds

`keep_only()` accepts a single mixed selector argument. Here we keep `"cmpd_a"` by name and the third sorted compound by index (`2`).

In [ ]:
subset = data.keep_only(["cmpd_a", 2])
print(subset.compounds)
print(subset.metadata)
subset.summary()

## Remove selected compounds

`remove()` uses the same selector rules and also preserves the original metadata.

In [ ]:
trimmed = data.remove("cmpd_b")
print(trimmed.compounds)
print(trimmed.metadata)
trimmed.summary()

## Concatenate datasets

Now we make a second dataset. It extends `cmpd_a` with a new independent experiment (`exp2`) and adds a new compound `cmpd_d`. `concatenate()` drops metadata on purpose, because merging metadata implicitly would be ambiguous.

In [ ]:
raw_extension = pd.DataFrame(
    make_rows("cmpd_a", "exp2", {0.1: [94.0, 93.0], 1.0: [46.0, 45.0]})
    + make_rows("cmpd_d", "exp1", {0.1: [88.0, 87.0], 1.0: [36.0, 35.0]})
)

data_extension = bc.DoseResponseData.from_dataframe(
    raw_extension,
    metadata={"source": "extension"},
)

merged = bc.DoseResponseData.concatenate(data, data_extension)
print(merged.compounds)
print(merged.metadata)
merged.summary()

## Round-trip to wide format and JSON

The wide-format round-trip preserves the assay values, even though replicate IDs are regenerated from the wide column names. The JSON round-trip in long format preserves the full canonical schema.

In [ ]:
wide_df = merged.to_dataframe(format="wide")
wide_df

In [ ]:
wide_roundtrip = bc.DoseResponseData.from_dataframe(wide_df, format="wide")
json_text = merged.to_json(indent=2)
json_roundtrip = bc.DoseResponseData.from_json(json_text)

value_cols = ["compound_id", "experiment_id", "concentration", "response"]
wide_match = merged.table[value_cols].sort_values(value_cols).reset_index(drop=True).equals(
    wide_roundtrip.table[value_cols].sort_values(value_cols).reset_index(drop=True)
)
json_match = merged.table.sort_values(value_cols + ["replicate_id"]).reset_index(drop=True).equals(
    json_roundtrip.table.sort_values(value_cols + ["replicate_id"]).reset_index(drop=True)
)

print(f"Wide round-trip preserves assay values: {wide_match}")
print(f"JSON long-form round-trip is exact: {json_match}")
print(json_text[:220] + "...")

## Strictness and conflict protection

The manipulation helpers are intentionally strict: bad selectors raise immediately, and `concatenate()` refuses to merge the same compound if that would reuse an existing `experiment_id`.

In [ ]:
for description, operation in [
    ("unknown compound", lambda: data.keep_only("missing")),
    ("out-of-range index", lambda: data.remove(10)),
    (
        "conflicting concatenate",
        lambda: bc.DoseResponseData.concatenate(
            data.keep_only("cmpd_a"),
            data.keep_only("cmpd_a"),
        ),
    ),
]:
    try:
        operation()
    except Exception as exc:
        print(f"{description}: {type(exc).__name__}: {exc}")

## Takeaways

`DoseResponseData` now gives you a small, strict data-manipulation layer before fitting:

- `keep_only()` for compound subsetting
- `remove()` for compound exclusion
- `concatenate()` for combining compatible datasets
- `to_dataframe()` and `to_json()` for simple round-tripping